# Exploration & Auswertung

Dieses Notebook visualisiert die Ergebnisse aus der Pipeline (`classify.py`).
Hier kannst du Plots vergleichen, Hypothesen testen und experimentieren.

**Voraussetzung:** `python classify.py` muss vorher gelaufen sein.

## Ergebnisse laden

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Pipeline-Ergebnisse laden
with open('Data/tfidf_results.pkl', 'rb') as f:
    res = pickle.load(f)

top_whisper = res['top_whisper']
top_ipa = res['top_ipa']
tfidf_whisper = res['tfidf_whisper']
tfidf_ipa = res['tfidf_ipa']
regions = sorted(top_whisper.keys())

# Bereinigte Daten laden (fuer eigene Experimente)
df = pd.read_csv('Data/transcriptions_clean.csv')

print(f'Geladen: {len(df):,} Zeilen, {len(regions)} Regionen')
print(f'Regionen: {regions}')
print(f'Spalten: {list(df.columns)}')

## 3.1a – Top-20 TF-IDF Woerter (Whisper Hochdeutsch)

In [ ]:
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860', '#DA8BC3']

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, region in enumerate(sorted(top_whisper)):
    terms = [w[0] for w in top_whisper[region]][::-1]
    scores = [w[1] for w in top_whisper[region]][::-1]
    axes[i].barh(terms, scores, color=colors[i])
    axes[i].set_title(region, fontweight='bold')
    axes[i].set_xlabel('TF-IDF Score')

axes[-1].set_visible(False)
fig.suptitle('Top-20 TF-IDF Woerter pro Region (Whisper Hochdeutsch)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.1b – Top-20 TF-IDF N-Grams (IPA bereinigt, ohne Betonungszeichen)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, region in enumerate(sorted(top_ipa)):
    terms = [f'"{w[0]}"' for w in top_ipa[region]][::-1]
    scores = [w[1] for w in top_ipa[region]][::-1]
    axes[i].barh(terms, scores, color=colors[i])
    axes[i].set_title(region, fontweight='bold')
    axes[i].set_xlabel('TF-IDF Score')

axes[-1].set_visible(False)
fig.suptitle('Top-20 TF-IDF Character N-Grams pro Region (IPA bereinigt)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Vergleich: Whisper vs. IPA

Schneller Ueberblick – welche Top-3 Features hat jede Region?

In [ ]:
comparison = []
for region in regions:
    w_top3 = ", ".join(w[0] for w in top_whisper[region][:3])
    i_top3 = ", ".join(f'"{w[0]}"' for w in top_ipa[region][:3])
    comparison.append({"Region": region, "Whisper Top-3": w_top3, "IPA Top-3": i_top3})

pd.DataFrame(comparison).set_index("Region")

## Gleiche Woerter, andere Aussprache

Welche Woerter uebersetzt Whisper fuer **alle Regionen gleich** (= selbes Hochdeutsch-Wort),
aber das IPA-Modell schreibt sie **unterschiedlich** (= andere Aussprache)?

Das sind die spannendsten Faelle: Gleicher Inhalt, aber der Dialekt klingt anders!

In [ ]:
from collections import Counter

# --- Schritt 1: Whisper-Woerter pro Region sammeln ---
# Fuer jede Region: welche Woerter kommen vor?
region_words = {}
for region in regions:
    texts = df[df['dialect_region'] == region]['transcription_whisper'].str.lower().str.split()
    all_words = [w for words in texts for w in words]
    region_words[region] = Counter(all_words)

# --- Schritt 2: Woerter finden die in ALLEN Regionen vorkommen ---
# (mind. 3x pro Region, damit es kein Zufall ist)
common_words = set(region_words[regions[0]].keys())
for region in regions[1:]:
    common_words &= set(region_words[region].keys())

# Nur Woerter die in jeder Region mind. 3x vorkommen
min_count = 3
common_words = {w for w in common_words
                if all(region_words[r][w] >= min_count for r in regions)}

print(f"Woerter die in allen {len(regions)} Regionen mind. {min_count}x vorkommen: {len(common_words)}")
print(f"Beispiele: {sorted(common_words)[:20]}")

In [ ]:
# --- Schritt 3: Fuer diese Woerter die IPA-Varianten pro Region sammeln ---
# Idee: Wenn Whisper "nicht" schreibt, wie klingt "nicht" in jeder Region laut IPA?
# Wir suchen Saetze die das Wort enthalten und schauen, welche IPA-Strings darin vorkommen.

# Funktion: IPA-Kontext um ein Wort herum extrahieren
# Da Whisper und IPA nicht wort-aligniert sind, nehmen wir den ganzen IPA-String
# und vergleichen die IPA-Strings von Saetzen die das gleiche Whisper-Wort enthalten.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

word_divergence = {}

for word in common_words:
    # Fuer jede Region: IPA-Strings der Saetze sammeln die dieses Wort enthalten
    region_ipa_texts = {}
    for region in regions:
        mask = (df['dialect_region'] == region) & \
               (df['transcription_whisper'].str.lower().str.contains(rf'\b{word}\b', regex=True, na=False))
        ipa_texts = df.loc[mask, 'ipa_clean'].tolist()
        if len(ipa_texts) >= 2:
            region_ipa_texts[region] = ' '.join(ipa_texts)

    # Nur Woerter die in mind. 5 Regionen Treffer haben
    if len(region_ipa_texts) < 5:
        continue

    # IPA-Aehnlichkeit zwischen den Regionen berechnen
    try:
        vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=500)
        region_names = list(region_ipa_texts.keys())
        tfidf = vec.fit_transform(region_ipa_texts.values())
        sim_matrix = cosine_similarity(tfidf)

        # Durchschnittliche paarweise Aehnlichkeit (ohne Diagonale)
        n = len(region_names)
        pairwise = [sim_matrix[i][j] for i in range(n) for j in range(i+1, n)]
        avg_sim = np.mean(pairwise)

        word_divergence[word] = {
            'avg_similarity': avg_sim,
            'n_regions': len(region_ipa_texts),
            'min_sim': min(pairwise),
            'max_sim': max(pairwise),
        }
    except Exception:
        continue

print(f"Woerter mit genug Daten fuer den Vergleich: {len(word_divergence)}")

In [ ]:
# --- Schritt 4: Top-20 Woerter mit groesster IPA-Divergenz ---
# Sortiert nach niedrigster Aehnlichkeit = groesste Dialektunterschiede

divergence_df = pd.DataFrame(word_divergence).T
divergence_df = divergence_df.sort_values('avg_similarity')

print("Top-20 Woerter: gleiches Hochdeutsch, aber am unterschiedlichsten ausgesprochen")
print("=" * 75)
print(f"{'Wort':<20} {'Avg Sim':>8} {'Min Sim':>8} {'Max Sim':>8} {'Regionen':>9}")
print("-" * 75)
for word, row in divergence_df.head(20).iterrows():
    print(f"{word:<20} {row['avg_similarity']:>8.3f} {row['min_sim']:>8.3f} {row['max_sim']:>8.3f} {int(row['n_regions']):>9}")

In [ ]:
# --- Schritt 5: Visualisierung – Top-10 divergenteste Woerter ---

fig, ax = plt.subplots(figsize=(12, 6))
top10 = divergence_df.head(10)

bars = ax.barh(
    top10.index[::-1],
    (1 - top10['avg_similarity'].values)[::-1],
    color='#C44E52', edgecolor='black', linewidth=0.5
)
ax.set_xlabel('IPA-Divergenz (1 - Cosine Similarity)')
ax.set_title('Top-10 Woerter: Gleiches Hochdeutsch, unterschiedlichste Aussprache',
             fontweight='bold', fontsize=13)

# Werte an Balken
for bar, val in zip(bars, (1 - top10['avg_similarity'].values)[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Schritt 6: Detail-Ansicht – Wie klingt das divergenteste Wort pro Region? ---
# Fuer die Top-3 divergentesten Woerter: ein IPA-Beispiel pro Region zeigen

top3_words = divergence_df.head(3).index.tolist()

for word in top3_words:
    print(f'\n{"=" * 60}')
    print(f'Whisper-Wort: "{word}"')
    print(f'{"=" * 60}')
    for region in regions:
        mask = (df['dialect_region'] == region) & \
               (df['transcription_whisper'].str.lower().str.contains(rf'\b{word}\b', regex=True, na=False))
        samples = df.loc[mask]
        if len(samples) > 0:
            row = samples.iloc[0]
            # Whisper-Satz kuerzen
            whisper_short = row['transcription_whisper'][:60]
            ipa_short = row['ipa_clean'][:60]
            print(f'  {region:15s}  HD: {whisper_short}...')
            print(f'  {"":15s} IPA: {ipa_short}...')
        else:
            print(f'  {region:15s}  (kein Treffer)')

## Experimentier-Bereich

Hier kannst du eigene Analysen machen, z.B.:
- Andere N-Gram Ranges testen
- Einzelne Regionen vergleichen
- IPA-Strings manuell anschauen

In [ ]:
# Beispiel: IPA-Strings einer Region anschauen
region = 'Wallis'
samples = df[df['dialect_region'] == region].head(5)
for _, row in samples.iterrows():
    print(f"Whisper: {row['transcription_whisper']}")
    print(f"IPA:     {row['ipa_clean']}")
    print()

### Vergleich: Whisper vs. IPA

**Fragen die wir beantworten wollen:**
- Zeigen die IPA-Features **mehr Unterschiede** zwischen den Regionen als die Whisper-Features?
- Gibt es Regionen die sich stärker abheben (z.B. Wallis)?
- Sind die Whisper Top-Wörter eher generisch (der, die, das) oder regionsspezifisch?